# snRNA + ATAC joint embedding with WNN (Trevino et al. 2021)

In [ ]:
import os
from pathlib import Path
import scanpy as sc
import muon as mu

DATA_ROOT = Path(os.environ.get('B_CELL_ATLAS_DATA', 'data'))
sc.set_figure_params(figsize=[6, 6], color_map='viridis_r')

## Load preprocessed objects and build MuData

In [ ]:
rna = sc.read_h5ad(DATA_ROOT / 'Trevino_RNA_preprocessed.h5ad')
atac = sc.read_h5ad(DATA_ROOT / 'Trevino_ATAC_preprocessed.h5ad')
# verify cells are matched between modalities
assert all(rna.obs_names == atac.obs_names)
mdata = mu.MuData({'rna': rna, 'atac': atac})
mdata.var_names_make_unique()
print(mdata)

## Compare individual UMAP embeddings

In [ ]:
mdata.update_obs()
mdata.obsm['X_umap_rna'] = mdata['rna'].obsm['X_umap'].copy()
mdata.obsm['X_umap_atac'] = mdata['atac'].obsm['X_umap'].copy()
sc.pl.embedding(mdata, basis='X_umap_rna', color='rna:cluster_name', legend_loc='on data', title='RNA UMAP')
sc.pl.embedding(mdata, basis='X_umap_atac', color='rna:cluster_name', legend_loc='on data', title='ATAC UMAP')

## WNN joint embedding

In [ ]:
# WNN uses per-modality KNN graphs already stored in rna.obsp and atac.obsp
mu.pp.neighbors(mdata)
mu.tl.umap(mdata)
mdata.obsm['X_umap_wnn'] = mdata.obsm['X_umap'].copy()
mu.pl.embedding(mdata, basis='X_umap_wnn', color=['rna:cluster_name'],
               legend_loc='on data', title='WNN joint UMAP')

## Modality weights

In [ ]:
# cells with high RNA weight rely more on transcriptomic signal;
# proliferating cells often show elevated RNA weight due to ATAC noise
mu.pl.embedding(mdata, basis='X_umap_wnn',
               color=['rna:mod_weight', 'atac:mod_weight'],
               cmap='cividis')

## Save MuData

In [ ]:
mdata.write(DATA_ROOT / 'Trevino_multiome_wnn.h5mu')